In [1]:
import polars as pl
import numpy as np

from scipy import stats

from ohlc_dss_model.data import load_parquet, remove_incomplete_days, intraday_session_tagging, session_tagging
from ohlc_dss_model.utils import convert_to_timezone

Next is we need to model volatility in our features, problem is since we only have OHLC data and not tick data we will essentially need to estimate it, there are a couple of model such as Close to close, Parkinsons, Rogers Satchell, Yang Zhang.

Lets look at Close to close first as it is the most simplest model out of all, it measures volatility based on closing time of each day, which means what happens between those closing times doesn't really matter, this is a huge flaw as our model relies heavily on intraday movement so this model is out of the question

Another thing is we really are not supposed to calculate sigma_yz_t to include Target session at all since if we were to include Target session in the calculation at all it means we will be leaking the Target volatility into our features

In [2]:
df = load_parquet()
df = convert_to_timezone(df)
df = session_tagging(df)
df = intraday_session_tagging(df)
df = remove_incomplete_days(df)

print(df.head(5))

shape: (5, 8)
┌──────────────────┬─────────┬─────────┬─────────┬─────────┬────────┬────────────┬─────────────────┐
│ DateTime         ┆ Open    ┆ High    ┆ Low     ┆ Close   ┆ Volume ┆ Session    ┆ Intraday_Sessio │
│ ---              ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---    ┆ ---        ┆ n               │
│ datetime[ns, Ame ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ u64    ┆ date       ┆ ---             │
│ rica/New_York]   ┆         ┆         ┆         ┆         ┆        ┆            ┆ str             │
╞══════════════════╪═════════╪═════════╪═════════╪═════════╪════════╪════════════╪═════════════════╡
│ 2011-01-02       ┆ 2220.0  ┆ 2223.75 ┆ 2219.25 ┆ 2220.5  ┆ 889    ┆ 2011-01-03 ┆ Pre_Target_1    │
│ 18:00:00 EST     ┆         ┆         ┆         ┆         ┆        ┆            ┆                 │
│ 2011-01-02       ┆ 2220.0  ┆ 2222.0  ┆ 2219.75 ┆ 2221.5  ┆ 238    ┆ 2011-01-03 ┆ Pre_Target_1    │
│ 18:30:00 EST     ┆         ┆         ┆         ┆         ┆        ┆        

We will then aggregate the session OHLC such that one row represents one trading day

In [3]:
features = (
    df.group_by(["Session", "Intraday_Session"])
    .agg([
        pl.col("Open").first().alias("O"),
        pl.col("High").max().alias("H"),
        pl.col("Low").min().alias("L"),
        pl.col("Close").last().alias("C"),
    ])
    
    .pivot(
        index="Session",
        on="Intraday_Session",
        values=["O", "H", "L", "C"],
    )
    .sort("Session")
)
print(features.head(5))
print(features.shape)
print(features.columns)

shape: (5, 17)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Session   ┆ O_Target_ ┆ O_Pre_Tar ┆ O_Pre_Tar ┆ … ┆ C_Target_ ┆ C_Pre_Tar ┆ C_Pre_Tar ┆ C_Target │
│ ---       ┆ 2         ┆ get_2     ┆ get_1     ┆   ┆ 2         ┆ get_2     ┆ get_1     ┆ _1       │
│ date      ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2011-01-0 ┆ 2258.25   ┆ 2226.75   ┆ 2220.0    ┆ … ┆ 2250.75   ┆ 2239.25   ┆ 2227.25   ┆ 2263.5   │
│ 3         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 2011-01-0 ┆ 2242.0    ┆ 2251.25   ┆ 2251.25   ┆ … ┆ 2247.25   ┆ 2257.0    ┆ 2251.25   ┆ 2241.75  │
│ 4         ┆           ┆           ┆           ┆   ┆           ┆           

One of the differences between these estimators is that some of them such as Rogers-Satchell and Parkinson ignores overnight gaps, now we need to determine if overnight gaps are material or not

In [4]:
gap = features.with_columns([
    ((pl.col("O_Pre_Target_1") / pl.col("C_Target_2").shift(1)).log()).alias("Overnight_Gap"),
    ((pl.col("O_Pre_Target_1") / pl.col("C_Pre_Target_1")).log()).alias("Pre_Target_1_Return"),
    ((pl.col("C_Pre_Target_2") / pl.col("O_Pre_Target_2")).log()).alias("Pre_Target_2_Return"),
]).drop_nulls(["Overnight_Gap"])

for col, label in [
    ("Overnight_Gap", "Overnight Gap (C_NY -> O_Asia)"),
    ("Pre_Target_1_Return", "Pre_Target_1 O to C return"),
    ("Pre_Target_2_Return", "Pre_Target_2 to C return"),
]:
    vals = gap[col].to_numpy()
    print(f"{label}")
    print(f"mean abs: {np.abs(vals).mean():.6f}")
    print(f"std: {vals.std():.6f}")
    print(f"p95 abs: {np.percentile(np.abs(vals), 95):.6f}")
    print(f"max abs: {np.abs(vals).max():.6f}")
    print()

Overnight Gap (C_NY -> O_Asia)
mean abs: 0.000980
std: 0.002702
p95 abs: 0.003851
max abs: 0.058147

Pre_Target_1 O to C return
mean abs: 0.002974
std: 0.004872
p95 abs: 0.009259
max abs: 0.049560

Pre_Target_2 to C return
mean abs: 0.003017
std: 0.004562
p95 abs: 0.009139
max abs: 0.057329



Lets look at the mean, although overnight gaps have a lower mean compared to all of the session we also see that standard deviation are quite high, this means data are noisy and a coefficient of variation of ~ 3.3x meaning there will be time where gaps will be huge, take a look at max abs, overnight gaps have the largest, this can be caused by either news or covid for example, this is an example why we cannot really ignore this gaps, so the only estimator that fits our case are Yang-Zhang estimator.

Parkinsons assumes zero drift, but we need to see if asia or london does have drift at all if so then it violates this assumption.

In [5]:
drift = features.with_columns([
    ((pl.col("C_Pre_Target_1") / pl.col("O_Pre_Target_1")).log()).alias("Drift_Pre_Target_1"),
    ((pl.col("C_Pre_Target_2") / pl.col("O_Pre_Target_2")).log()).alias("Drift_Pre_Target_2"),
])

for session, col in [("Pre_Target_1", "Drift_Pre_Target_1"), ("Pre_Target_2", "Drift_Pre_Target_2")]:
    vals = drift[col].drop_nulls().to_numpy()
    t_stat, p_value = stats.ttest_1samp(vals, popmean=0)
    print(f"{session} session drift")
    print(f"mean: {vals.mean():.6f}")
    print(f"std: {vals.std():.6f}  (cross-day drift variance)")
    print(f"t-statistic: {t_stat:.4f}")
    print(f"p-value: {p_value:.4f}")
    print(f"pct days |drift| > 0.001: {(np.abs(vals) > 0.001).mean()*100:.1f}%")
    print()

Pre_Target_1 session drift
mean: 0.000293
std: 0.004871  (cross-day drift variance)
t-statistic: 3.6889
p-value: 0.0002
pct days |drift| > 0.001: 70.2%

Pre_Target_2 session drift
mean: 0.000121
std: 0.004562  (cross-day drift variance)
t-statistic: 1.6330
p-value: 0.1025
pct days |drift| > 0.001: 72.9%



Although both mean drift on asia and london are not statistically significant, standard deviation tells more to the story with 70.2% and 72.9% of both Asia and London exhibit absolute drift more day 0.1% this violates the assumption of zero drift for Parkinsons.

Hence our choice for the volatility estimator would be Yang Zhang estimator